In [ ]:
# ================================================================
# CELLA 0 — Setup: Drive + codice da GitHub + dipendenze
# Esegui OGNI VOLTA che apri il notebook.
# GPU: Runtime -> Cambia tipo di runtime -> T4 GPU
# ================================================================
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

REPO_URL = 'https://github.com/ruggiio/polimi-notes.git'
CODE_DIR = '/content/polimi-notes'                  # codice: fresco da GitHub
DATA_DIR = '/content/drive/MyDrive/polimi-notes'    # dati persistenti su Drive

# 1) Codice sempre aggiornato da GitHub
if os.path.exists(CODE_DIR):
    !git -C {CODE_DIR} pull --ff-only
else:
    !git clone --depth 1 {REPO_URL} {CODE_DIR}
%cd {CODE_DIR}

# 2) Dati persistenti su Drive, collegati dentro al repo con symlink
#    (output/ contiene anche RAG, archivio corso e PDF -> tutto resta su Drive)
for d in ['output', 'registrazioni', 'materiale_corso']:
    os.makedirs(f'{DATA_DIR}/{d}', exist_ok=True)
    if not os.path.islink(d):
        if os.path.exists(d):
            shutil.rmtree(d)
        os.symlink(f'{DATA_DIR}/{d}', d)

# 3) Schede corso persistenti su Drive (config/courses)
os.makedirs(f'{DATA_DIR}/courses', exist_ok=True)
if not os.path.islink('config/courses'):
    if os.path.exists('config/courses/_template.md') and not os.path.exists(f'{DATA_DIR}/courses/_template.md'):
        shutil.copy2('config/courses/_template.md', f'{DATA_DIR}/courses/_template.md')
    shutil.rmtree('config/courses', ignore_errors=True)
    os.symlink(f'{DATA_DIR}/courses', 'config/courses')

# 4) Dipendenze python (NON reinstallare torch: teniamo la build CUDA di Colab)
!grep -v '^torch' requirements.txt > /tmp/req.txt
!pip install -q -r /tmp/req.txt

# 5) ffmpeg + LaTeX (texlive-latex-extra serve per tcolorbox/titlesec del preambolo)
!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq ffmpeg texlive-latex-base texlive-fonts-recommended texlive-latex-extra

# 6) Smoke test: se questo compila, ogni lezione compila
!cd tests && pdflatex -interaction=nonstopmode preamble_smoke.tex > /dev/null 2>&1 && echo '[OK] Preambolo LaTeX: compila.' || echo '[ERRORE] Il preambolo NON compila!'

import torch
print('\n[OK] Ambiente pronto.')
print('CUDA disponibile:', torch.cuda.is_available(), '| versione CUDA:', torch.version.cuda)
if not torch.cuda.is_available():
    print('[WARN] GPU non vista -> Runtime -> Cambia tipo di runtime -> T4 GPU, poi Riavvia sessione.')

In [ ]:
# ================================================================
# CELLA 1 — Impostazioni: modifica QUI e basta
# ================================================================
from google.colab import userdata
import os

CORSO = 'Model Order'            # nome corso (= nome usato nel RAG)
RIFAI_ESISTENTI = False          # True = rigenera anche le lezioni gia' fatte

os.environ['ANTHROPIC_API_KEY'] = userdata.get('polimi-notes')
os.environ['POLIMI_COURSE'] = CORSO   # aggancia la scheda corso (glossario Whisper)

DATA_DIR = '/content/drive/MyDrive/polimi-notes'
print(f'Corso: {CORSO}')
scheda = f"config/courses/{CORSO.lower().replace(' ', '_')}.md"
print('Scheda corso:', scheda, '->', 'TROVATA' if os.path.exists(scheda) else 'assente (opzionale: copia _template.md)')

In [ ]:
# ================================================================
# CELLA 2 — Elabora i video caricati in registrazioni/ su Drive
# trascrizione -> note -> archivio corso (+ PDF se compile_pdf: true)
# ================================================================
import os, re, glob, shutil, datetime

def trova_data(nome, percorso):
    """Ricava la data della lezione dal nome file; fallback: data del file."""
    m = re.search(r'(20\d{2})[-_]?(\d{2})[-_]?(\d{2})', nome)
    if m:
        return f"{m.group(1)}-{m.group(2)}-{m.group(3)}"
    m = re.search(r'(\d{2})[-_](\d{2})[-_](20\d{2})', nome)
    if m:
        return f"{m.group(3)}-{m.group(2)}-{m.group(1)}"
    return datetime.date.fromtimestamp(os.path.getmtime(percorso)).isoformat()

ESTENSIONI = ('.mp4', '.mkv', '.webm', '.m4v', '.mov', '.avi')
video = sorted({v for ext in ESTENSIONI for v in glob.glob(f'registrazioni/**/*{ext}', recursive=True)})

if not video:
    print("[!] Nessun video in 'registrazioni/'. Caricali su Drive e riesegui.")

ok = saltate = fallite = 0
for v in video:
    nome = os.path.basename(v)
    stem = os.path.splitext(nome)[0]
    data = trova_data(nome, v)
    backup_dir = f"{DATA_DIR}/salvataggi_{CORSO.replace(' ', '_')}/lezione_{stem}"

    print(f"\n{'='*55}\n LEZIONE: {nome}  (data {data})\n{'='*55}")
    if not RIFAI_ESISTENTI and glob.glob(f'{backup_dir}/latex/lecture_notes.tex'):
        print('  [OK] Gia\' elaborata, salto.')
        saltate += 1
        continue

    os.makedirs(backup_dir, exist_ok=True)
    rc = os.system(
        f'python main.py run "dummy_url" --course "{CORSO}" '
        f'--no-download --no-ocr --date "{data}" --video "{v}"'
    )
    if rc != 0:
        print(f'  [ERRORE] (exit {rc}). Continuo col prossimo.')
        fallite += 1
        continue

    for subdir in ['output/transcripts', 'output/latex']:
        if os.path.exists(subdir):
            shutil.copytree(subdir, f"{backup_dir}/{subdir.split('/')[1]}", dirs_exist_ok=True)
    pdfs = sorted(glob.glob('output/notes/*.pdf'), key=os.path.getmtime, reverse=True)
    if pdfs:
        shutil.copy2(pdfs[0], backup_dir + '/')
        print(f'  PDF: {os.path.basename(pdfs[0])}')
    ok += 1
    print(f"  [OK] Fatto -> archivio corso aggiornato (output/course/)")

print(f"\n{'='*55}\n Elaborate: {ok}   Saltate: {saltate}   Fallite: {fallite}\n{'='*55}")

In [ ]:
# ================================================================
# CELLA 3 — (una tantum) Importa nell'archivio corso le lezioni
# elaborate PRIMA che esistesse l'archivio (dai salvataggi su Drive)
# ================================================================
import glob, os, re, shutil

slug = re.sub(r'[^a-z0-9]+', '_', CORSO.lower()).strip('_')
dest = f'output/course/{slug}'
os.makedirs(dest, exist_ok=True)

n = 0
for tex in glob.glob(f"{DATA_DIR}/salvataggi_{CORSO.replace(' ', '_')}/lezione_*/latex/lecture_notes.tex"):
    lezdir = tex.split('/')[-3]
    m = re.search(r'(20\d{2})[-_](\d{2})[-_](\d{2})', lezdir)
    if not m:
        print('[skip] data non riconosciuta:', lezdir)
        continue
    out = f"{dest}/{m.group(1)}-{m.group(2)}-{m.group(3)}_{lezdir}.tex"
    if not os.path.exists(out):
        shutil.copy2(tex, out)
        n += 1
        print('[OK]', os.path.basename(out))

print(f'\nImportate {n} lezioni. Archivio attuale:')
for f in sorted(glob.glob(f'{dest}/*.tex')):
    print('  -', os.path.basename(f))

In [ ]:
# ================================================================
# CELLA 4 — FINE CORSO: genera il PDF unico e coeso
# Eseguila SOLO quando il corso e' concluso.
# POLISH=True: passata Claude per capitolo (transizioni, notazione
# uniforme, titoli descrittivi). False: fusione meccanica, gratis.
# ================================================================
POLISH = True

flag = '' if POLISH else '--no-polish'
rc = os.system(f'python -m src.course_builder "{CORSO}" {flag}')
print('\n[OK] Vedi output/notes/ su Drive.' if rc == 0 else f'[ERRORE] exit {rc}')